# Qwen3-4B-Instruct-2507 — Sinhala QA Test Split Inference

Context-grounded evaluation of the **base** `Qwen/Qwen3-4B-Instruct-2507` on `test.jsonl`.

This is the hallucination-mitigation baseline: the model is never fine-tuned here, it only
receives the gold context in the prompt. The question this notebook answers is *how faithful
is Qwen3-4B to a supplied Sinhala context?*

Mirrors `llama-scripts/test-split-inference.ipynb` (same test file location, same reporting
flow) with the changes the model requires:

- **Chat model, not a completion model.** Prompts go through the Qwen chat template with a
  system message instead of the raw `"පිළිතුර: ["` completion trick.
- **Refusal string comes from the dataset.** The gold refusal in `test.jsonl` is used as the
  canonical `NO_ANSWER`, so abstentions can be scored by exact match. Alternative refusal
  phrasings the model produces are detected and normalised.
- **Larger token budget.** Qwen's BPE has no Sinhala vocabulary, so Sinhala costs far more
  tokens per character than it does on the Sinhala-extended Llama tokenizer.
- **Metrics match `qa-finetuning_v6.ipynb`** (normalised exact match, token F1 with a digit
  guard, evidence support, false-answer rate) so results are directly comparable to
  `llama_model_answers/*-v6-results.txt`.
- **Optional lexical grounding gate**, the same one v6 uses: an ungrounded generation is
  converted to a refusal. Raw and gated scores are both reported, which isolates how much
  hallucination the gate actually removes.

In [ ]:
%uv pip install -q "transformers>=4.51.0" accelerate safetensors hf_transfer

In [ ]:
import json
import os
import re
import unicodedata
from collections import Counter
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

SEED = 42
set_seed(SEED)

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

# Same locations as llama-scripts/test-split-inference.ipynb: the file is uploaded to /tmp on
# Modal. The repo-relative paths are fallbacks so the notebook also runs locally.
TEST_FILE_CANDIDATES = [
    Path("/tmp/test.jsonl"),
    Path("/tmp/test_updated.jsonl"),
    Path("new_split_v2/test.jsonl"),
    Path("../new_split_v2/test.jsonl"),
    Path("splits/test.jsonl"),
    Path("../splits/test.jsonl"),
]
TEST_FILE = next((path for path in TEST_FILE_CANDIDATES if path.is_file()), TEST_FILE_CANDIDATES[0])

OUTPUT_FILE = Path("qwen3-4b-instruct-2507-test-predictions.jsonl")
RESULTS_TXT = Path("qwen3-4b-instruct-2507-results.txt")

# Canonical refusal. This is the gold string used in the test split, so it must match exactly
# or every unanswerable row scores zero.
NO_ANSWER = "මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත."

# Other refusal phrasings an instruct model drifts into. Any of these is treated as an
# abstention and rewritten to NO_ANSWER before scoring.
REFUSAL_MARKERS = [
    "ප්‍රමාණවත් තොරතුරු නොමැත",
    "පිළිතුර නොමැත",
    "පිළිතුරු නොමැත",
    "සඳහන් වී නොමැත",
    "සඳහන් නොවේ",
    "දක්වා නොමැත",
    "not enough information",
    "no answer",
    "cannot be found",
]

# Qwen has no Sinhala vocabulary and falls back to byte-level BPE, so Sinhala costs far more
# tokens here than on the Sinhala-extended Llama tokenizer. Measured on this test split, gold
# answers run to a median of 53 tokens and a maximum of 261 — the 48-80 token budget used in
# the Llama notebooks would silently truncate the long ones. The cell that loads the data
# re-checks this against the actual file.
MAX_NEW_TOKENS = 320
MAX_INPUT_TOKENS = 16384
BATCH_SIZE = 8

# Grounding gate, same semantics as qa-finetuning_v6.ipynb.
USE_GROUNDING = True
GROUNDING_THRESHOLD = 0.50

# "greedy" for reproducible evaluation. "qwen_sampling" is Qwen's own recommended preset for
# this checkpoint, kept for sensitivity checks — do not report headline numbers from it.
DECODING = "greedy"

print("Test file :", TEST_FILE)
print("Exists    :", TEST_FILE.is_file())
print("Model     :", MODEL_ID)
print("Decoding  :", DECODING)

In [ ]:
# Only needed if a gated or private repo is used. Requires HF_TOKEN in the environment.
# from huggingface_hub import login
# login(token=os.environ["HF_TOKEN"])

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this 4B model.")

model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("GPU:", torch.cuda.get_device_name(0))
print("Loading:", MODEL_ID)
print("Dtype:", model_dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Batched generation requires left padding, otherwise the pad run sits between the prompt and
# the first generated token and the model continues from padding.
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=model_dtype,
    device_map="auto",
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
model.eval()
model.config.pad_token_id = tokenizer.pad_token_id

print("Model ready on:", model.device)

In [ ]:
def apply_decoding_preset(name):
    """Write the decoding preset onto model.generation_config.

    Qwen ships do_sample=True with temperature/top_p/top_k in generation_config.json. Leaving
    those set while generating greedily produces a warning on every call and is easy to
    misread later, so the sampling fields are cleared explicitly.
    """
    config = model.generation_config
    if name == "greedy":
        config.do_sample = False
        config.temperature = None
        config.top_p = None
        config.top_k = None
        config.min_p = None
    elif name == "qwen_sampling":
        # Qwen's published settings for Qwen3-*-Instruct-2507.
        config.do_sample = True
        config.temperature = 0.7
        config.top_p = 0.8
        config.top_k = 20
        config.min_p = 0.0
    else:
        raise ValueError(f"Unknown decoding preset: {name}")

    # A light penalty stops the degenerate Sinhala token loops greedy decoding falls into
    # without distorting short extractive answers.
    config.repetition_penalty = 1.05
    config.pad_token_id = tokenizer.pad_token_id
    return config


generation_config = apply_decoding_preset(DECODING)
EOS_IDS = generation_config.eos_token_id
EOS_IDS = set(EOS_IDS) if isinstance(EOS_IDS, (list, tuple)) else {EOS_IDS}

print("Decoding preset :", DECODING)
print("do_sample       :", generation_config.do_sample)
print("temperature     :", generation_config.temperature)
print("repetition_pen. :", generation_config.repetition_penalty)
print("eos_token_id    :", sorted(EOS_IDS))
print("max_new_tokens  :", MAX_NEW_TOKENS)

In [ ]:
SYSTEM_PROMPT = f"""You are a helpful Sinhala history question-answering assistant.

Your task is to answer the question using ONLY the information explicitly provided in the context.

Instructions:

- Read the entire context carefully before answering.
- Use only the information explicitly stated in the context.
- Do not use external knowledge, assumptions, or prior knowledge.
- Identify the exact information requested by the question.
- If the answer is found in multiple parts of the context, combine the relevant information into a single complete answer.
- Include only information that directly answers the question.
- Do not include additional facts, names, dates, or events unless they are required to answer the question.
- Match the person or entity named in the question exactly.
- Use evidence that contains both the requested entity and the requested attribute.
- Do not take a date or fact from a neighboring sentence about a different entity or event.
- Do not infer or guess information that is not explicitly stated, except for simple arithmetic explicitly requested by the question when all required values are stated in the context.
- For a duration question with explicit starting and ending years, subtract the starting year from the ending year and return the duration.
- If the answer cannot be found in the context, respond exactly with:
  "{NO_ANSWER}"
- Return only the final answer in natural Sinhala.
- Do not explain your reasoning.
- Do not mention passage numbers, page numbers, chapter names, grades, or any other source references.
- Answer in a single line. Do not add a preamble, a label, or quotation marks."""


def build_messages(context, question):
    user_prompt = f"""Context:

{context}

Question:

{question}

Answer:"""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


def preview_prompt(context, question, tail=1200):
    rendered = tokenizer.apply_chat_template(
        build_messages(context, question),
        add_generation_prompt=True,
        tokenize=False,
    )
    token_count = len(tokenizer(rendered, add_special_tokens=False)["input_ids"])
    print("=== Rendered prompt tail ===")
    print(rendered[-tail:])
    print("=== Prompt tokens ===")
    print(token_count)
    return token_count


print("Prompt builder ready.")

In [ ]:
# Text handling shared with qa-finetuning_v6.ipynb so scores are computed identically.

SINHALA_WORD_RE = re.compile(r"[\w඀-෿]+", re.UNICODE)
STOPWORDS = {
    "හා", "සහ", "හෝ", "දී", "ද", "ය", "යි", "වේ", "විය", "වූ", "ලෙස",
    "විසින්", "සඳහා", "සිට", "දක්වා", "එම", "මෙම", "ඒ", "ඔහු", "ඇය",
    "කුමක්ද", "කවුද", "කවදාද", "කෙසේද", "කොපමණද", "මොනවාද",
}


def clean_text(value):
    text = unicodedata.normalize("NFC", str(value or ""))
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


def lexical_tokens(value):
    tokens = [token.casefold() for token in SINHALA_WORD_RE.findall(clean_text(value))]
    return [token for token in tokens if len(token) >= 2 and token not in STOPWORDS]


def token_supported(token, normalized_context):
    if token in normalized_context:
        return True
    # Sinhala case endings often add one character; a short stem check keeps grounded variants.
    return len(token) >= 4 and token[:-1] in normalized_context


def normalize_answer(value):
    text = clean_text(value).casefold()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" \t\r\n[]{}()<>\"'`.,!?;:।෴")


def is_no_answer(value):
    normalized = normalize_answer(value)
    if not normalized:
        return True
    if normalized == normalize_answer(NO_ANSWER):
        return True
    return any(marker.casefold() in normalized for marker in REFUSAL_MARKERS)


def evidence_support(answer, context):
    """Fraction of the answer's content tokens that appear in the context.

    This is the hallucination proxy: 1.0 means every content word is traceable to the
    supplied context, low values mean the model brought in outside knowledge.
    """
    answer_tokens = lexical_tokens(answer)
    if not answer_tokens:
        return 0.0
    normalized_context = " ".join(lexical_tokens(context))
    supported = sum(token_supported(token, normalized_context) for token in answer_tokens)
    return supported / len(answer_tokens)


def token_f1(prediction, reference):
    prediction_digits = re.findall(r"\d+", normalize_answer(prediction))
    reference_digits = re.findall(r"\d+", normalize_answer(reference))
    # A wrong year is a wrong answer even when the surrounding words overlap.
    if reference_digits and prediction_digits != reference_digits:
        return 0.0
    prediction_tokens = lexical_tokens(prediction)
    reference_tokens = lexical_tokens(reference)
    if not prediction_tokens and not reference_tokens:
        return 1.0
    if not prediction_tokens or not reference_tokens:
        return 0.0
    common = Counter(prediction_tokens) & Counter(reference_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(prediction_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


def keyword_match(prediction, gold):
    """Soft match carried over from test-split-inference.ipynb, for continuity."""
    pred = normalize_answer(prediction)
    target = normalize_answer(gold)
    if not pred or not target:
        return False
    if pred == target or target in pred or pred in target:
        return True
    target_tokens = [token for token in target.split() if len(token) > 1]
    if not target_tokens:
        return False
    matched = sum(1 for token in target_tokens if token in pred)
    return matched / len(target_tokens) >= 0.6


print("Scoring helpers ready.")

In [ ]:
def canonical_answer(item):
    if item.get("answerable") is False:
        return NO_ANSWER
    answer = clean_text(item.get("answer", ""))
    return answer if answer else NO_ANSWER


def load_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Required JSONL file not found: {path}")

    records = []
    fingerprints = set()
    dropped = 0
    duplicates = 0

    with path.open("r", encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: {error}") from error

            question = clean_text(item.get("question"))
            context = clean_text(item.get("context"))
            answerable = item.get("answerable")
            if type(answerable) is not bool:
                answerable = bool(clean_text(item.get("answer")))

            normalized = {
                "question": question,
                "context": context,
                "answer": clean_text(item.get("answer")),
                "answerable": answerable,
                "grade": item.get("grade"),
                "chapter": item.get("chapter"),
                "chapter_title": item.get("chapter_title"),
            }
            if not question or not context or (answerable and not normalized["answer"]):
                dropped += 1
                continue

            fingerprint = (normalized["question"], normalized["context"], normalized["answer"])
            if fingerprint in fingerprints:
                duplicates += 1
                continue
            fingerprints.add(fingerprint)
            records.append(normalized)

    return records, dropped, duplicates


test_rows, dropped_rows, duplicate_rows = load_jsonl(TEST_FILE)
LOAD_HEADER = [
    f"Loaded {len(test_rows)} unique records from {TEST_FILE}",
    f"Dropped invalid/empty: {dropped_rows}; exact duplicates removed: {duplicate_rows}",
]
for line in LOAD_HEADER:
    print(line)
print(Counter(row["answerable"] for row in test_rows))

# Confirm MAX_NEW_TOKENS actually covers the gold answers under Qwen's tokenizer.
gold_lengths = [
    len(tokenizer(canonical_answer(row), add_special_tokens=False)["input_ids"])
    for row in test_rows
]
gold_lengths.sort()
print(
    "Gold answer tokens (Qwen BPE) - median:",
    gold_lengths[len(gold_lengths) // 2],
    "| p95:",
    gold_lengths[int(len(gold_lengths) * 0.95)],
    "| max:",
    gold_lengths[-1],
)
print("MAX_NEW_TOKENS:", MAX_NEW_TOKENS)
if gold_lengths[-1] > MAX_NEW_TOKENS:
    print("WARNING: raise MAX_NEW_TOKENS, the longest gold answer does not fit.")

In [ ]:
LABEL_PREFIX_RE = re.compile(r"^\s*(පිළිතුර|answer)\s*[:：]\s*", re.IGNORECASE)


def postprocess(text):
    answer = clean_text(text)
    if not answer:
        return ""
    # Instruct models occasionally emit a label or a preamble line before the answer.
    answer = answer.splitlines()[0]
    answer = LABEL_PREFIX_RE.sub("", answer)
    return answer.strip(" []{}()<>\"'`*").strip()


def generate_raw(pairs, max_new_tokens=MAX_NEW_TOKENS, batch_size=BATCH_SIZE):
    """Batched greedy generation over (context, question) pairs.

    Returns one dict per pair with the decoded answer, its input token count and whether the
    generation hit the token cap (a truncated answer scores badly for the wrong reason).
    """
    results = []

    for start in range(0, len(pairs), batch_size):
        chunk = pairs[start : start + batch_size]
        conversations = [build_messages(context, question) for context, question in chunk]

        inputs = tokenizer.apply_chat_template(
            conversations,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        ).to(model.device)

        input_length = inputs["input_ids"].shape[-1]
        if input_length > MAX_INPUT_TOKENS:
            raise ValueError(
                f"Prompt has {input_length:,} tokens, exceeding the evaluation limit of "
                f"{MAX_INPUT_TOKENS:,}. Retrieve or rerank fewer context passages."
            )

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                generation_config=generation_config,
                max_new_tokens=max_new_tokens,
                use_cache=True,
            )

        generated = output_ids[:, input_length:]
        attention = inputs["attention_mask"]

        for row_index in range(generated.shape[0]):
            row_ids = generated[row_index].tolist()
            hit_cap = len(row_ids) >= max_new_tokens and not any(
                token in EOS_IDS for token in row_ids
            )
            text = tokenizer.decode(row_ids, skip_special_tokens=True)
            results.append({
                "raw_answer": postprocess(text),
                "input_tokens": int(attention[row_index].sum().item()),
                "truncated": hit_cap,
            })

    return results


def run_qa(context, question, use_grounding=USE_GROUNDING):
    """Single-row convenience wrapper with the grounding gate applied."""
    generated = generate_raw([(context, question)])[0]
    return apply_grounding(generated, context, use_grounding=use_grounding)


def apply_grounding(generated, context, use_grounding=USE_GROUNDING):
    raw_answer = generated["raw_answer"]
    # An empty generation is a decoding failure, not a deliberate abstention. It is still
    # scored as a refusal, but tracked separately so it cannot be read as good behaviour.
    empty = not raw_answer
    refused = not empty and is_no_answer(raw_answer)
    support = 0.0 if empty else (1.0 if refused else evidence_support(raw_answer, context))

    if empty or refused:
        final_answer = NO_ANSWER
        gated = False
    elif use_grounding and support < GROUNDING_THRESHOLD:
        # The generation is not traceable to the context: treat it as a hallucination and
        # abstain instead.
        final_answer = NO_ANSWER
        gated = True
    else:
        final_answer = raw_answer
        gated = False

    return {
        **generated,
        "answer": final_answer,
        "support": support,
        "refused_directly": refused,
        "empty_generation": empty,
        "gated": gated,
        "verbatim_in_context": bool(raw_answer)
        and not refused
        and normalize_answer(raw_answer) in normalize_answer(context),
    }


print("Inference helpers ready.")

In [ ]:
# Smoke test on one row before committing to the full split.
row = test_rows[0]
preview_prompt(row["context"], row["question"])

result = run_qa(row["context"], row["question"])
print()
print("Question  :", row["question"])
print("Reference :", canonical_answer(row))
print("Raw       :", result["raw_answer"])
print("Final     :", result["answer"])
print("Support   :", f"{result['support']:.3f}")
print("Truncated :", result["truncated"])

In [ ]:
pairs = [(row["context"], row["question"]) for row in test_rows]
generations = []

for start in range(0, len(pairs), BATCH_SIZE):
    generations.extend(generate_raw(pairs[start : start + BATCH_SIZE], batch_size=BATCH_SIZE))
    done = min(start + BATCH_SIZE, len(pairs))
    print(f"Done {done}/{len(pairs)}", flush=True)

print("Finished generation.")

In [ ]:
predictions = []

for index, (row, generated) in enumerate(zip(test_rows, generations), start=1):
    reference = canonical_answer(row)
    result = apply_grounding(generated, row["context"], use_grounding=USE_GROUNDING)

    prediction = result["answer"]
    raw_prediction = result["raw_answer"]

    exact = normalize_answer(prediction) == normalize_answer(reference)
    raw_exact = normalize_answer(raw_prediction) == normalize_answer(reference)
    f1 = token_f1(prediction, reference)
    raw_f1 = token_f1(raw_prediction, reference)

    predictions.append({
        "index": index,
        "grade": row.get("grade"),
        "chapter": row.get("chapter"),
        "answerable": row["answerable"],
        "context": row["context"],
        "question": row["question"],
        "reference": reference,
        "raw_prediction": raw_prediction,
        "prediction": prediction,
        "exact_match": exact,
        "raw_exact_match": raw_exact,
        "token_f1": f1,
        "raw_token_f1": raw_f1,
        "soft_match": keyword_match(prediction, reference),
        "evidence_support": result["support"],
        "predicted_no_answer": is_no_answer(prediction),
        "refused_directly": result["refused_directly"],
        "empty_generation": result["empty_generation"],
        "gated": result["gated"],
        "verbatim_in_context": result["verbatim_in_context"],
        "input_tokens": result["input_tokens"],
        "truncated": result["truncated"],
    })

print("Scored", len(predictions), "rows.")

In [ ]:
total = len(predictions)
answerable = [item for item in predictions if item["answerable"]]
unanswerable = [item for item in predictions if not item["answerable"]]

exact_correct = sum(item["exact_match"] for item in predictions)
raw_exact_correct = sum(item["raw_exact_match"] for item in predictions)
soft_correct = sum(item["soft_match"] for item in predictions)
f1_total = sum(item["token_f1"] for item in predictions)
raw_f1_total = sum(item["raw_token_f1"] for item in predictions)

answerable_correct = sum(item["exact_match"] for item in answerable)
unanswerable_correct = sum(item["exact_match"] for item in unanswerable)
false_answers = len(unanswerable) - unanswerable_correct
over_abstentions = sum(item["predicted_no_answer"] for item in answerable)

predicted_no_answer_count = sum(item["predicted_no_answer"] for item in predictions)
correct_no_answer_count = sum(
    item["predicted_no_answer"] and not item["answerable"] for item in predictions
)
no_answer_precision = correct_no_answer_count / max(predicted_no_answer_count, 1)
no_answer_recall = correct_no_answer_count / max(len(unanswerable), 1)
no_answer_f1 = (
    2 * no_answer_precision * no_answer_recall / (no_answer_precision + no_answer_recall)
    if no_answer_precision + no_answer_recall
    else 0.0
)

attempted = [item for item in predictions if not item["predicted_no_answer"]]
mean_support_attempted = (
    sum(item["evidence_support"] for item in attempted) / len(attempted) if attempted else 0.0
)
low_support_attempted = sum(
    item["evidence_support"] < GROUNDING_THRESHOLD for item in attempted
)
gated_count = sum(item["gated"] for item in predictions)
verbatim_count = sum(item["verbatim_in_context"] for item in predictions)
truncated_count = sum(item["truncated"] for item in predictions)
empty_count = sum(item["empty_generation"] for item in predictions)


def percentage(part, whole):
    return 100 * part / max(whole, 1)


SUMMARY = [
    "=" * 100,
    f"TEST RESULTS — {MODEL_ID}",
    "=" * 100,
    f"Test file            : {TEST_FILE}",
    f"Decoding             : {DECODING} (repetition_penalty={generation_config.repetition_penalty}, "
    f"max_new_tokens={MAX_NEW_TOKENS})",
    f"Grounding gate       : {'on' if USE_GROUNDING else 'off'} (threshold={GROUNDING_THRESHOLD})",
    "",
    f"Total                : {total}  (answerable={len(answerable)}, unanswerable={len(unanswerable)})",
    f"Grounded exact match : {exact_correct}/{total} ({percentage(exact_correct, total):.2f}%)",
    f"Raw exact match      : {raw_exact_correct}/{total} ({percentage(raw_exact_correct, total):.2f}%)",
    f"Soft match           : {soft_correct}/{total} ({percentage(soft_correct, total):.2f}%)",
    f"Mean token F1        : {f1_total / max(total, 1):.4f}",
    f"Mean raw token F1    : {raw_f1_total / max(total, 1):.4f}",
    f"Answerable exact     : {answerable_correct}/{len(answerable)} "
    f"({percentage(answerable_correct, len(answerable)):.2f}%)",
    f"Unanswerable exact   : {unanswerable_correct}/{len(unanswerable)} "
    f"({percentage(unanswerable_correct, len(unanswerable)):.2f}%)",
    "",
    "--- Hallucination / abstention behaviour ---",
    f"False-answer rate on unanswerable : {false_answers}/{len(unanswerable)} "
    f"({percentage(false_answers, len(unanswerable)):.2f}%)",
    f"Over-abstention on answerable     : {over_abstentions}/{len(answerable)} "
    f"({percentage(over_abstentions, len(answerable)):.2f}%)",
    f"No-answer precision/recall/F1     : {no_answer_precision:.4f} / {no_answer_recall:.4f} / "
    f"{no_answer_f1:.4f}",
    f"Mean evidence support (attempted) : {mean_support_attempted:.4f} over {len(attempted)} rows",
    f"Ungrounded attempts (support<{GROUNDING_THRESHOLD}) : {low_support_attempted}",
    f"Generations rejected by the gate  : {gated_count}",
    f"Answers copied verbatim from ctx  : {verbatim_count}/{total} "
    f"({percentage(verbatim_count, total):.2f}%)",
    f"Generations hitting the token cap : {truncated_count}",
    f"Empty generations (scored as refusal) : {empty_count}",
    "=" * 100,
]

for line in SUMMARY:
    print(line)

In [ ]:
with OUTPUT_FILE.open("w", encoding="utf-8", newline="\n") as handle:
    for item in predictions:
        handle.write(json.dumps(item, ensure_ascii=False) + "\n")

# Same layout as llama_model_answers/*-v6-results.txt so the two models can be diffed directly.
with RESULTS_TXT.open("w", encoding="utf-8", newline="\n") as handle:
    for line in LOAD_HEADER:
        handle.write(line + "\n")

    for item in predictions:
        block = [
            "",
            "=" * 100,
            f"[{item['index']}/{total}]",
            f"Question : {item['question']}",
            f"Reference: {item['reference']}",
            f"Raw      : {item['raw_prediction']}",
            f"Final    : {item['prediction']}",
            f"Support  : {item['evidence_support']:.3f}",
            f"Exact/F1 : {item['exact_match']} / {item['token_f1']:.3f}",
        ]
        handle.write("\n".join(block) + "\n")

    handle.write("\n" + "\n".join(SUMMARY) + "\n")

print("Saved predictions to", OUTPUT_FILE.resolve())
print("Saved report to", RESULTS_TXT.resolve())

In [ ]:
# Hallucinations: the model answered an unanswerable question instead of abstaining.
hallucinated = [
    item for item in predictions if not item["answerable"] and not item["predicted_no_answer"]
]
print(f"Fabricated answers on unanswerable questions: {len(hallucinated)}")

for item in hallucinated[:10]:
    print("-" * 100)
    print("Question :", item["question"])
    print("Context  :", item["context"][:300])
    print("Raw      :", item["raw_prediction"])
    print("Final    :", item["prediction"])
    print("Support  :", f"{item['evidence_support']:.3f}")

In [ ]:
# Ungrounded content: answered, but the wording is not traceable to the context.
ungrounded = [
    item
    for item in predictions
    if not item["predicted_no_answer"] and item["evidence_support"] < GROUNDING_THRESHOLD
]
print(f"Low-support answers: {len(ungrounded)}")

for item in ungrounded[:10]:
    print("-" * 100)
    print("Question :", item["question"])
    print("Reference:", item["reference"])
    print("Raw      :", item["raw_prediction"])
    print("Support  :", f"{item['evidence_support']:.3f}")

In [ ]:
# Wrong-but-confident answers on questions that do have a context answer.
failed = [item for item in predictions if item["answerable"] and item["token_f1"] < 0.5]
print(f"Answerable rows below 0.5 token F1: {len(failed)}")

for item in failed[:10]:
    print("-" * 100)
    print("Question :", item["question"])
    print("Reference:", item["reference"])
    print("Raw      :", item["raw_prediction"])
    print("Final    :", item["prediction"])
    print(f"Support  : {item['evidence_support']:.3f} | F1: {item['token_f1']:.3f}")

In [ ]:
# Full transcript of generated vs expected answers.
for item in predictions:
    print("=" * 100)
    print(f"[{item['index']}/{total}]")
    print(f"Grade    : {item.get('grade')} | Chapter: {item.get('chapter')}")
    print(f"Answerable: {item['answerable']}")
    print(f"Question : {item['question']}")
    print(f"Reference: {item['reference']}")
    print(f"Raw      : {item['raw_prediction']}")
    print(f"Final    : {item['prediction']}")
    print(f"Support  : {item['evidence_support']:.3f}")
    print(f"Exact/F1 : {item['exact_match']} / {item['token_f1']:.3f}")